In [ ]:
import time
start_time = time.time()

import pandas as pd
import torch
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


In [ ]:
if torch.backends.mps.is_available():
    device = "mps"
    pipeline_device = torch.device("mps")
else:
    device = "cpu"
    pipeline_device = -1

model_name = "textattack/distilbert-base-uncased-MRPC"
batch_size = 64
max_length = 128

clf = pipeline(
    task="text-classification",
    model=model_name,
    tokenizer=model_name,
    device=pipeline_device,
    truncation=True,
    padding=True,
    max_length=max_length
)

print(f"Selected device: {device}")
print(f"Loaded model: {model_name}")
print(f"Batch size: {batch_size}")
print(f"Max length: {max_length}")


In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print(f"Validation examples: {len(dataset)}")

preview_df = dataset.select(range(min(5, len(dataset)))).to_pandas()
print(preview_df[["sentence1", "sentence2", "label"]].to_string(index=False))


In [ ]:
pair_texts = [f"{s1} [SEP] {s2}" for s1, s2 in zip(dataset["sentence1"], dataset["sentence2"])]

raw_outputs = clf(pair_texts, batch_size=batch_size)

label_to_id = {str(k).upper(): int(v) for k, v in clf.model.config.label2id.items()}
predictions = []
for output in raw_outputs:
    label_name = output["label"].upper()
    if label_name in label_to_id:
        pred = label_to_id[label_name]
    elif label_name.endswith("0"):
        pred = 0
    elif label_name.endswith("1"):
        pred = 1
    else:
        raise ValueError(f"Unrecognized label returned by pipeline: {output['label']}")
    predictions.append(pred)

true_labels = list(dataset["label"])
print(f"Completed inference for {len(predictions)} examples.")


In [ ]:
accuracy = accuracy_score(true_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels,
    predictions,
    average="binary",
    zero_division=0
)
cm = confusion_matrix(true_labels, predictions)

results_df = pd.DataFrame([
    {
        "model_name": model_name,
        "dataset": "glue/mrpc",
        "split": "validation",
        "num_examples": len(dataset),
        "batch_size": batch_size,
        "max_length": max_length,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "device": device
    }
])

print(results_df.to_string(index=False))
print("Confusion Matrix:")
print(cm)


In [ ]:
examples_df = dataset.to_pandas()[["sentence1", "sentence2", "label"]].copy()
examples_df = examples_df.rename(columns={"label": "true_label"})
examples_df["predicted_label"] = predictions
examples_df["correct"] = examples_df["true_label"] == examples_df["predicted_label"]

print(examples_df.head(10).to_string(index=False))

mismatches_df = examples_df[~examples_df["correct"]].copy()
print(f"\nMismatches: {len(mismatches_df)}")
if len(mismatches_df) > 0:
    sample_errors_df = mismatches_df.head(10)
    print(sample_errors_df[["sentence1", "sentence2", "true_label", "predicted_label"]].to_string(index=False))

elapsed_seconds = time.time() - start_time
print(f"Total runtime (seconds): {elapsed_seconds:.2f}")
